# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros — IA 5.2 Computer Vision.

Alumno : Porcelli Fabricio

## 1. Explicacion completa del pipeline

El sistema implementa un pipeline incremental de Computer Vision que permite detectar y clasificar razas de perros a partir de una imagen. El flujo completo se compone de cuatro etapas que evolucionan en complejidad:

### Etapa 1: Embeddings y Busqueda por Similitud
1. El usuario sube una imagen a la interfaz Gradio.
2. La imagen se preprocesa (redimensionamiento a 224x224, conversion a tensor y normalizacion con media/std de ImageNet).
3. Se extrae un embedding de 512 dimensiones utilizando el backbone de ResNet18 pre-entrenado en ImageNet (sin la capa fully-connected).
4. El embedding se compara contra todos los registros de la base vectorial (almacenados en PostgreSQL+pgvector o alternativamente en un archivo JSON).
5. La similitud se calcula con cosine similarity o distancia L2, segun la configuracion.
6. Se retornan las top-K imagenes mas similares y se predice la raza mediante voto ponderado por score entre los vecinos.
7. Si el score del mejor vecino esta por debajo de un umbral configurable (SIMILARITY_THRESHOLD), se retorna "unknown" para evitar clasificaciones forzadas.

### Etapa 2: Clasificacion Supervisada
1. Se entrenan dos modelos de clasificacion sobre el dataset de 70 razas:
   - **ResNet18 fine-tuned**: backbone congelado (solo se entrena la FC head), aprovechando las features universales de ImageNet.
   - **CNN Custom**: arquitectura propia de 5 bloques convolucionales entrenada desde cero.
2. Ambos modelos usan el mismo pipeline de preprocesamiento y data augmentation.
3. La aplicacion permite seleccionar dinamicamente el modelo a utilizar para la clasificacion.

### Etapa 3: Deteccion y Clasificacion
1. La imagen ingresada se procesa con YOLOv8n, detector de objetos pre-entrenado en COCO.
2. Se filtran las detecciones correspondientes a la clase "dog" (COCO class ID 16) con confianza >= 0.25.
3. Para cada perro detectado, se recorta la region (bounding box) y se clasifica con el modelo ResNet18 fine-tuned de la Etapa 2.
4. Los resultados se renderizan sobre la imagen original: bounding boxes, etiquetas de raza y scores de confianza.

### Integracion de componentes
El sistema sigue una arquitectura de microservicios orquestada con Docker Compose:
- **Backend (FastAPI, puerto 8000)**: expone endpoints REST para busqueda, clasificacion y deteccion. Utiliza un patron asyncrono (HTTP 202 + polling) para tareas largas.
- **Frontend (Gradio + FastAPI, puerto 8081)**: interfaz de usuario con 3 pestanias correspondientes a cada etapa, mas una pestania de estado para consultar resultados.
- **PostgreSQL + pgvector**: base de datos vectorial para almacenar y recuperar embeddings eficientemente. Alternativamente se puede usar un almacenamiento JSON plano.
- **TaskManager**: gestiona el ciclo de vida de trabajos asincronos en memoria.

Los servicios se conectan mediante inyeccion de dependencias en `bootstrap.py`: SimilarityService depende de la base vectorial, DetectionService depende de ClassifierService, y la API utiliza los tres servicios.

## 2. Dataset

Se utiliza el **70 Dog Breeds Image Dataset** (Kaggle, gpiosenka/70-dog-breedsimage-data-set), que contiene 9346 imagenes distribuidas en 70 razas de perros.

### Splits

| Split  | Imagenes | Clases |
|--------|----------|--------|
| Train  | 7946     | 70     |
| Valid  | 700      | 70     |
| Test   | 700      | 70     |

### Distribucion por clase (train)
- Media: 113.5 imagenes por raza
- Minimo: 65 (American Hairless)
- Maximo: 198 (Shih-Tzu)
- Desvio estandar: 24.5
- Ratio max/min: 3.05 — dataset relativamente balanceado

### Consistencia entre splits
Las 70 clases estan presentes en train, valid y test, lo que garantiza que todas las razas sean evaluables.

### Calidad de las imagenes
- Sin imagenes corruptas (0 de 9346).
- Sin imagenes en escala de grises.
- Sin extensiones de archivo inesperadas.
- Todas las imagenes tienen dimension 224x224.

### Conjunto independiente de evaluacion
Se descargaron 20 imagenes externas desde la API dog.ceo (almacenadas en `data/test_external/`) para evaluar el comportamiento del pipeline completo con imagenes reales no pertenecientes al dataset original. Adicionalmente se creo una imagen compuesta con multiples perros para probar deteccion multi-objeto.

## 3. Preprocesamiento

### Transformaciones aplicadas

Todas las imagenes atraviesan el siguiente pipeline antes de ingresar a cualquier modelo (extractor de embeddings o clasificador):

1. **Resize a 224x224**: es el tamano de entrada estandar de ResNet18 y de la mayoria de los modelos de torchvision pre-entrenados en ImageNet. Todas las imagenes del dataset ya vienen en este tamano, pero se aplica por si el usuario sube una imagen de dimensiones arbitrarias.

2. **ToTensor**: convierte el array numpy (H, W, C) con valores uint8 [0, 255] a un tensor PyTorch (C, H, W) con valores float32 [0.0, 1.0].

3. **Normalize con media y std de ImageNet**: centra y escala cada canal usando mean=[0.485, 0.456, 0.406] y std=[0.229, 0.224, 0.225]. Esto es obligatorio porque los pesos pre-entrenados fueron optimizados con esa normalizacion.

### Data Augmentation (solo training)

Para reducir overfitting y mejorar la generalizacion, se aplican las siguientes transformaciones aleatorias durante el entrenamiento:
- **RandomHorizontalFlip (p=0.5)**: simula imagenes espejadas, comun en fotografia.
- **RandomRotation (15 grados)**: tolera pequeñas rotaciones de la camara.
- **RandomAffine (translate=0.1, scale=(0.9, 1.1))**: simula desplazamientos y pequenos cambios de escala.
- **ColorJitter (brillo=0.2, contraste=0.2, saturacion=0.2)**: expone al modelo a variaciones de iluminacion.

### Transformaciones NO aplicadas (y por que)
- **CLAHE / ecualizacion del histograma**: no se aplican porque ResNet18 fue entrenado con imagenes RGB sin ecualizar. Agregar ecualizacion modificaria la distribucion de color que el modelo aprendio a reconocer.
- **Blur / denoising**: las imagenes del dataset tienen calidad aceptable y no presentan ruido excesivo.

Ambos modelos (ResNet18 y CNN custom) usan exactamente el mismo pipeline de preprocesamiento. La unica diferencia es la arquitectura.

## 4. Justificacion de los modelos elegidos

### ResNet18 (baseline, Etapa 1)
Se eligio ResNet18 pre-entrenado en ImageNet como extractor de embeddings baseline por:
- **Arquitectura probada**: ResNet introduce conexiones residuales que evitan la degradacion del gradiente, permitiendo entrenar redes profundas de forma estable.
- **Pre-entrenamiento en ImageNet**: el modelo ya reconoce features visuales de alto nivel (bordes, texturas, formas, objetos) que son transferibles a la tarea de razas de perros.
- **Embeddings de 512 dimensiones**: balance entre poder representativo y costo computacional. Suficiente para separar 70 razas manteniendo la base vectorial en un tamano manejable (103 MB para 7946 imagenes).
- **Velocidad de inferencia**: ResNet18 es liviano (11.7M parametros, ~44 MB), permitiendo extraer embeddings en ~5ms por imagen en GPU.

### ResNet18 fine-tuned (Modelo A, Etapa 2)
Se realizo fine-tuning sobre ResNet18 con el backbone congelado por las siguientes razones:
- **Backbone congelado**: con solo ~100 imagenes por raza, fine-tunear los 11M parametros del backbone causaria overfitting. Congelar el backbone preserva las features universales de ImageNet y solo entrena la FC head (260K parametros).
- **Rapida convergencia**: alcanza 93.4% de accuracy en solo 5 epocas.
- **Consistencia con Etapa 1**: los embeddings son identicos a los del baseline, manteniendo compatibilidad con la base vectorial.

### CNN Custom (Modelo B, Etapa 2)
Disenada como alternativa ligera y reproducible desde cero. Si bien el modelo final a utilizar para la clasificación es la resnet18 fine-tuned, este modelo sirve para comparar otra alternativa y valorar las ventajas de resnet sobre una CNN cruda sin bloques residuales ni mayores modificaciones:
- **Arquitectura**: 5 bloques Conv2D+BatchNorm+ReLU+MaxPool (32→64→128→256→512 canales), AdaptiveAvgPool, Dropout(0.3), 2 capas FC (512→128→70).
- **Ventaja**: checkpoint de solo 6.2 MB (7x mas liviano que ResNet18), entrenamiento 100% reproducible sin dependencias externas.
- **Desventaja**: accuracy limitada (~45.1%) por ausencia de pre-entrenamiento, arquitectura sin conexiones residuales y volumen de datos insuficiente para entrenar desde cero 70 clases.

### YOLOv8n (Etapa 3)
Se selecciono YOLOv8n (nano) como detector de objetos por:
- **Pre-entrenamiento en COCO**: detecta 80 clases de objetos, incluyendo "dog" (class ID 16). No requiere entrenamiento adicional.
- **Velocidad**: la version nano es la mas rapida de la familia YOLOv8, adecuada para inferencia en tiempo real incluso en CPU.
- **Precision competitiva**: a pesar de ser la version mas pequena, YOLOv8n mantiene buena precision en deteccion de objetos de tamano medio-grande.

### Trade-offs

| Aspecto | ResNet18 FT | CNN Custom | YOLOv8n |
|---|---|---|---|
| Accuracy | 93.4% | 45.1% | — (deteccion) |
| Parametros | 11.7M (260K entrenables) | ~2.7M | 3.2M |
| Checkpoint | 43 MB | 6.2 MB | 6.2 MB |
| Pre-entrenamiento | ImageNet (SI) | No | COCO (SI) |
| Epocas de training | 5 | 20 | 0 (pre-entrenado) |
| Inferencia (GPU) | ~5ms | ~3ms | ~10ms |

## 5. Proceso de entrenamiento e hiperparametros

Ambos modelos se entrenaron en Google Colab con GPU Tesla T4 (15.6 GB VRAM).

### Hiperparametros comunes
- **Optimizador**: Adam (weight_decay=1e-4 para regularizacion L2)
- **Scheduler**: ReduceLROnPlateau (mode='min', factor=0.5, patience=5)
- **Loss function**: CrossEntropyLoss
- **DataLoader batch size**: 32
- **Class balancing**: WeightedRandomSampler con pesos = 1.0 / frecuencia de la clase, para que las razas minoritarias (American Hairless con 65 imagenes) se muestreen con mayor probabilidad.

### ResNet18 fine-tuned
- **Epocas**: 5
- **Learning rate**: 0.001
- **Backbone**: congelado (requires_grad=False), solo se entrena la FC head (nn.Linear(512, 70))
- **Progreso del entrenamiento**:
  - Epoca 1: Train Acc=60.7%, Val Acc=87.4%
  - Epoca 2: Train Acc=80.9%, Val Acc=91.7%
  - Epoca 3: Train Acc=84.3%, Val Acc=92.0%
  - Epoca 4: Train Acc=85.9%, Val Acc=92.1%
  - Epoca 5: Train Acc=86.8%, Val Acc=92.0%
- **Convergencia rapida**: la accuracy de validacion se estabiliza en ~92% desde la epoca 3, indicando que el backbone congelado ya provee features de alta calidad y solo se necesita ajustar el clasificador lineal.

### CNN Custom
- **Epocas**: 20
- **Learning rate**: 0.001
- **Entrenamiento desde cero**: todos los pesos se inicializan aleatoriamente
- **Arquitectura**:
  - Bloque 1: Conv2d(3,32) + BN + ReLU + MaxPool(2)
  - Bloque 2: Conv2d(32,64) + BN + ReLU + MaxPool(2)
  - Bloque 3: Conv2d(64,128) + BN + ReLU + MaxPool(2)
  - Bloque 4: Conv2d(128,256) + BN + ReLU + MaxPool(2)
  - Bloque 5: Conv2d(256,512) + BN + ReLU + MaxPool(2)
  - AdaptiveAvgPool(1,1) + Flatten
  - Dropout(0.3) + Linear(512,128) + ReLU + Linear(128,70)
- **Progreso**: accuracy final de 49.0% en train y 47.9% en validacion. La curva de entrenamiento muestra una mejora sostenida pero lenta, sin evidencia de overfitting significativo (train y validacion se mantienen cercanas).

## 6. Resultados obtenidos

### Etapa 1: Busqueda por similitud (ResNet18 baseline)

**NDCG@10 (Normalized Discounted Cumulative Gain):**
- Promedio: 0.8156
- Mediana: 0.9123
- Desvio estandar: 0.2292
- Minimo: 0.0000
- Maximo: 1.0000

**Justificacion:** el NDCG@10 promedio de 0.8156 indica que, en promedio, las imagenes relevantes (de la misma raza) aparecen bien rankeadas entre los top-10 resultados. La mediana alta (0.9123) muestra que mas de la mitad de las consultas tienen un ranking casi perfecto. El desvio alto (0.2292) se debe a que algunas razas visualmente similares (ej. Bulldog vs Boston Terrier) obtienen scores bajos. El valor minimo de 0.0 ocurre cuando ninguna de las 10 imagenes mas similares pertenece a la raza correcta, tipicamente en razas que se confunden sistematicamente con otra.

**Metricas de clasificacion por similitud:**
- Accuracy: 93.86%
- Precision: 94.38%
- Recall: 93.86%
- Specificity: 99.91%
- F1-Score: 93.70%

**Confusiones mas frecuentes (43 errores de 700 imagenes = 6.1%):**
- Bulldog -> Boston Terrier (4 veces)
- Bull Mastiff -> Boxer (3 veces)
- Labradoodle -> Cockapoo (3 veces)
- American Hairless -> Mex Hairless (2 veces)
- American Spaniel -> Irish Spaniel (2 veces)
- Boston Terrier -> Bulldog (2 veces)
- German Sheperd -> Malinois (2 veces)

Estas confusiones corresponden a razas morfologicamente similares (terriers, spaniels, molosos) donde las diferencias sutiles no son capturadas por el embedding de ResNet18 baseline.

### Etapa 2: Clasificacion supervisada

**Metricas en test (700 imagenes, 70 clases):**

| Metrica | CNN Custom | ResNet18 FT | Diferencia |
|---|---|---|---|
| Accuracy | 0.4514 | 0.9343 | +48.3% |
| Precision | 0.5370 | 0.9400 | +40.3% |
| Recall | 0.4514 | 0.9343 | +48.3% |
| Specificity | 0.9920 | 0.9990 | +0.7% |
| F1-Score | 0.4213 | 0.9337 | +51.2% |

**Mejores clases (ResNet18, accuracy 1.00):** Mex Hairless, Newfoundland, Pekinese, Rottweiler, Saint Bernard, Schnauzer, Shar_Pei, Siberian Husky, Vizsla, Yorkie.

**Peores clases (ResNet18):** Boston Terrier (0.60), Labrador (0.60), Cockapoo (0.70), Dingo (0.70), Malinois (0.70), Shih-Tzu (0.70).

**Matriz de confusion:** la matriz de confusion de ResNet18 muestra una diagonal dominante, con pocas confusiones fuera de ella. Las confusiones se concentran en pares de razas visualmente similares (ej. Bulldog/French Bulldog, Labradoodle/Cockapoo, German Sheperd/Malinois).

## 7. Comparacion entre enfoques

### Busqueda por similitud vs Clasificacion supervisada

| Aspecto | Busqueda por similitud (Etapa 1) | Clasificacion supervisada (Etapa 2) |
|---|---|---|
| Accuracy | 93.86% | 93.43% (ResNet18 FT) |
| Entrenamiento | No requiere | Requiere (~5 min en GPU) |
| Nuevas clases | Agregar imagenes a la DB | Requiere re-entrenar |
| Interpretabilidad | Muestra las imagenes mas similares | Solo da la etiqueta |
| Umbral de rechazo | SIMILARITY_THRESHOLD retorna "unknown" | Softmax score, menos robusto |
| Velocidad de inferencia | O(N) con N tamano de DB | O(1) (forward pass) |

La busqueda por similitud ofrece accuracy ligeramente superior (93.86% vs 93.43%), pero esto se debe a que el clasificador fine-tuned evalua sobre 70 clases mientras que la busqueda por similitud simplemente retorna el vecino mas cercano, que puede no ser la clase correcta aunque el ranking sea bueno (medido por NDCG@10). Ambos enfoques son complementarios: la busqueda por similitud es ideal para prototipado rapido y para incorporar nuevas clases sin re-entrenar, mientras que la clasificacion supervisada es mas adecuada para produccion donde se requiere una sola etiqueta con confianza.

### ResNet18 fine-tuned vs CNN Custom

| Metrica | ResNet18 FT | CNN Custom | Ganador |
|---|---|---|---|
| Accuracy | 0.934 | 0.451 | ResNet18 (+48.3%) |
| Precision | 0.940 | 0.537 | ResNet18 (+40.3%) |
| Recall | 0.934 | 0.451 | ResNet18 (+48.3%) |
| Specificity | 0.999 | 0.992 | Empate tecnico |
| F1-Score | 0.934 | 0.421 | ResNet18 (+51.2%) |
| Checkpoint | 43 MB | 6.2 MB | CNN (7x mas liviano) |
| Epocas | 5 | 20 | ResNet18 (4x mas rapido) |
| Pre-entrenamiento | Si (ImageNet, congelado) | No (desde cero) | — |

**Conclusion:** ResNet18 con backbone congelado es superior en todos los aspectos metricos. La diferencia se debe al pre-entrenamiento en ImageNet, que proporciona features visuales universales de alta calidad. La CNN custom queda como alternativa solo cuando el peso del checkpoint o la reproducibilidad absoluta son requisitos excluyentes. Una mejora posible para la CNN custom seria agregar conexiones residuales para permitir mayor profundidad sin degradacion del gradiente.

### Analisis de la Specificity
Ambos modelos obtienen specificity >0.99, lo que significa que rara vez clasifican una imagen como una raza que no es. Esto es esperable en clasificacion multiclase con 70 clases: la probabilidad de un falso positivo para una clase dada es baja porque hay 69 clases negativas.

## 8. Problemas encontrados y soluciones implementadas

### Problema 1: Overfitting en ResNet18 (Etapa 2)
- **Sintoma**: en la primera iteracion de entrenamiento, ResNet18 alcanzaba ~99% de accuracy en train pero solo ~60% en validacion (commit `f4631bc`).
- **Causa**: se estaban entrenando todos los parametros del modelo (11.7M), incluyendo el backbone pre-entrenado. Con solo ~100 imagenes por clase, el modelo memorizaba el conjunto de entrenamiento.
- **Solucion**: congelar el backbone (`requires_grad=False`) y entrenar solo la FC head (260K parametros) (commit `d8a1111`). Esto preserva las features de ImageNet y solo aprende la combinacion lineal para las 70 razas.
- **Resultado**: accuracy de validacion subio de ~60% a ~92%.

### Problema 2: Error en el nombre de una raza en el dataset
- **Sintoma**: la clase "American Spaniel" aparecia con doble espacio ("American  Spaniel") en el split `valid/`, causando un desajuste con `train/` y `test/`.
- **Solucion**: renombrar el directorio manualmente en la notebook (`mv data/dataset/valid/"American  Spaniel" data/dataset/valid/"American Spaniel"`).

### Problema 3: Desbalanceo de clases
- **Sintoma**: la raza American Hairless tiene 65 imagenes mientras que Shih-Tzu tiene 198 (ratio 3:1).
- **Solucion**: incorporar `WeightedRandomSampler` en `train_classifier` (commit `f6319f8`), con pesos inversamente proporcionales a la frecuencia de cada clase. Esto asegura que cada epoca vea un numero balanceado de ejemplos por raza, evitando que el modelo se sesgue hacia las clases mayoritarias.

### Problema 4: Estabilidad del entrenamiento de la CNN custom
- **Sintoma**: la CNN custom mostraba fluctuaciones en accuracy de validacion entre epocas consecutivas (ej. epoca 6: 19.6%, epoca 7: 18.3%, epoca 8: 29.0%).
- **Solucion**: incorporar BatchNormalization despues de cada capa convolucional para estabilizar la distribucion de las activaciones, y utilizar ReduceLROnPlateau para reducir el learning rate cuando la loss de validacion se estanca.

### Problema 5: YOLO no detectaba perros en las imágenes del dataset (Etapa 3)
- **Síntoma**: al probar YOLOv8n sobre las imágenes del dataset (splits train/valid/test), el detector no encontraba ningún perro en ninguna imagen.
- **Causa**: las imágenes del dataset tienen resolución 224×224, mientras que YOLOv8n está entrenado en COCO con imágenes de 640×640. El upscaling forzado de 224 a 640 píxeles degrada severamente la calidad de la imagen, impidiendo que el detector reconozca los patrones de la clase "dog" aprendidos durante su entrenamiento.
- **Solución**: se implementó el script scripts/external_images.py que descarga imágenes de 640×640 desde la API placedog.net. Estas imágenes externas tienen la resolución esperada por YOLO y el detector funciona correctamente en ellas. Las imágenes del dataset se siguen utilizando exclusivamente para las etapas 1 y 2 (extracción de embeddings y clasificación), que sí operan sobre 224×224.

### Problema 6: Clasificacion de recortes con baja confianza (Etapa 3)
- **Sintoma**: algunos recortes de YOLO con baja confianza de deteccion (ej. 0.22) producian clasificaciones con score bajo (ej. Bull Terrier 0.22).
- **Solucion**: las detecciones con confianza < YOLO_CONF_THRESHOLD (0.25) se filtran antes de clasificar. No se implemento un umbral adicional para la clasificacion, ya que el score del clasificador refleja naturalmente la incertidumbre cuando el recorte es de baja calidad.

## 9. Modificaciones fuera de las funciones indicadas

Las unicas modificaciones realizadas fuera de las funciones puntuales indicadas en cada etapa fueron:

1. **Agregar el script `scripts/external_images.py`**: permite descargar imagenes de prueba externas desde la API dog.ceo para evaluar el pipeline de deteccion (Etapa 3). No es parte de las funciones requeridas pero facilita la evaluacion del sistema.

2. **Agregar `extract_custom_embedding` en `classifier_service.py`**: esta funcion no estaba en la consigna original pero se implemento para que el frontend pueda usar los modelos de Etapa 2 como extractores de embeddings en la pestaña de busqueda por similitud (Etapa 1), permitiendo seleccionar dinamicamente entre baseline, ResNet18 fine-tuned y CNN custom.